# Feature Engineering

**Course:** [ML in Practice](https://ml-viz.vercel.app/courses/ml-in-practice/01-feature-engineering)

This notebook demonstrates feature scaling, categorical encoding strategies, detecting data leakage, and building leakage-safe sklearn Pipelines.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

np.random.seed(42)

plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#2a2d3a',
    'axes.labelcolor': '#e2e8f0',
    'text.color': '#e2e8f0',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'grid.color': '#2a2d3a',
    'grid.alpha': 0.5,
})

## Feature scaling: Standard vs MinMax vs Robust

In [ ]:
# Generate feature with outliers (income in thousands)
normal_incomes = np.random.normal(50, 15, 200)
outlier_incomes = np.array([200, 250, 300, 400])  # very high earners
X = np.concatenate([normal_incomes, outlier_incomes])

scalers = {
    'Original': None,
    'StandardScaler': StandardScaler(),
    'MinMaxScaler': MinMaxScaler(),
    'RobustScaler': RobustScaler(),
}

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (name, scaler) in zip(axes, scalers.items()):
    if scaler is None:
        data = X
    else:
        data = scaler.fit_transform(X.reshape(-1, 1)).ravel()
    ax.hist(data, bins=30, color='#6366f1', alpha=0.8, edgecolor='#4f46e5')
    # Mark outliers
    if scaler is None:
        out_vals = outlier_incomes
    else:
        out_vals = scaler.transform(outlier_incomes.reshape(-1, 1)).ravel()
    ax.axvline(out_vals.mean(), color='#ef4444', linestyle='--', alpha=0.8, label=f'Outlier mean')
    ax.set_title(name, fontsize=11)
    ax.set_xlabel('Value')
    if name != 'Original':
        ax.legend(fontsize=8, facecolor='#1a1d27', edgecolor='#2a2d3a')

plt.suptitle('Effect of scalers on income distribution with outliers', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print("\nRobustScaler is resistant to outliers:")
rs = RobustScaler().fit(X.reshape(-1, 1))
print(f"  Median: {rs.center_:.1f}, IQR: {rs.scale_:.1f}")

## Categorical encoding: comparing strategies

In [ ]:
import pandas as pd

# Simulate a dataset with a categorical feature and binary target
np.random.seed(0)
n = 1000
cities = ['New York', 'Los Angeles', 'Chicago', 'Houston', 'Phoenix']
city_rates = {'New York': 0.7, 'Los Angeles': 0.5, 'Chicago': 0.4, 'Houston': 0.3, 'Phoenix': 0.2}

city_col = np.random.choice(cities, n)
target = np.array([np.random.binomial(1, city_rates[c]) for c in city_col])

df = pd.DataFrame({'city': city_col, 'target': target})

# One-hot encoding
ohe = pd.get_dummies(df['city'], prefix='city')

# Frequency encoding
freq_enc = df['city'].map(df['city'].value_counts(normalize=True))

# Target encoding (DANGEROUS without cross-fold — showing the issue)
global_mean = df['target'].mean()
target_enc_naive = df.groupby('city')['target'].mean()
df['target_enc'] = df['city'].map(target_enc_naive)

print("Encoding comparison for 'city' feature:")
print("\nOne-hot (first 3 rows):")
print(pd.concat([df[['city', 'target']], ohe], axis=1).head(3).to_string())
print("\nTarget encoding (city → mean target):")
print(target_enc_naive.sort_values(ascending=False))

## Data leakage: the scaler trap

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

X_data, y_data = make_classification(n_samples=1000, n_features=20, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X_data, y_data, test_size=0.2, random_state=42)

# ❌ WRONG: fit scaler on full dataset before split
scaler_leaky = StandardScaler()
X_all_scaled = scaler_leaky.fit_transform(X_data)  # uses test set stats!
X_tr_leaky = X_all_scaled[:800]
X_te_leaky = X_all_scaled[800:]
clf_leaky = LogisticRegression(max_iter=500)
clf_leaky.fit(X_tr_leaky, y_tr)
acc_leaky = clf_leaky.score(X_te_leaky, y_te)

# ✅ CORRECT: fit scaler on training set only
pipe_correct = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=500))
])
pipe_correct.fit(X_tr, y_tr)  # scaler only sees X_tr
acc_correct = pipe_correct.score(X_te, y_te)

print(f"Leaky scaler (fit on full data):   {acc_leaky:.4f}")
print(f"Correct Pipeline (fit on train):   {acc_correct:.4f}")
print(f"Difference: {abs(acc_leaky - acc_correct):.4f} ({'leaky higher' if acc_leaky > acc_correct else 'correct higher'})")

## ✏️ Your turn

### Exercise 1: Detect target leakage

Given a feature matrix and target, identify which features are likely target leakage by checking temporal constraints.

In [ ]:
def detect_high_correlation_features(X, y, threshold=0.9):
    """
    Find features with suspiciously high correlation with the target.
    High correlation with target may indicate target leakage.
    
    Args:
        X: np.ndarray (n_samples, n_features)
        y: np.ndarray (n_samples,) binary target
        threshold: float, correlation threshold to flag as suspicious
    Returns:
        list of int: feature indices with |correlation| > threshold
    """
    # TODO(you): compute Pearson correlation between each feature and y
    # Return indices where |correlation| > threshold
    pass


# Create test data: 2 normal features + 1 leaky (highly correlated with target)
np.random.seed(5)
n = 200
y_test = np.random.binomial(1, 0.5, n)
X_test = np.column_stack([
    np.random.randn(n),          # feature 0: unrelated
    np.random.randn(n),          # feature 1: unrelated  
    y_test + np.random.randn(n) * 0.05,  # feature 2: leaky (almost = target)
])

leaky = detect_high_correlation_features(X_test, y_test, threshold=0.9)
print(f"Suspicious features (|corr| > 0.9): {leaky}")

In [ ]:
leaky = detect_high_correlation_features(X_test, y_test, threshold=0.9)
assert leaky is not None, "Should return a list"
assert 2 in leaky, "Feature index 2 (the leaky one) should be flagged"
assert 0 not in leaky, "Feature 0 (unrelated) should not be flagged"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def detect_high_correlation_features(X, y, threshold=0.9):
    suspicious = []
    y_centered = y - y.mean()
    for i in range(X.shape[1]):
        x_centered = X[:, i] - X[:, i].mean()
        corr = np.dot(x_centered, y_centered) / (
            np.linalg.norm(x_centered) * np.linalg.norm(y_centered) + 1e-8
        )
        if abs(corr) > threshold:
            suspicious.append(i)
    return suspicious
```
</details>

### Exercise 2: Implement frequency encoding

Replace each category with its frequency (proportion) in the training set.

In [ ]:
def frequency_encode(train_categories, test_categories):
    """
    Encode categories by their frequency in the training set.
    
    Args:
        train_categories: list/array of category labels (training set)
        test_categories: list/array of category labels (test set)
    Returns:
        tuple: (train_encoded, test_encoded) — arrays of float frequencies
               Unknown categories in test get frequency 0.0
    """
    # TODO(you): compute category frequencies from train_categories
    # Map train and test to those frequencies (unknown test categories → 0.0)
    pass


train_cats = ['A', 'B', 'A', 'C', 'A', 'B', 'C', 'A']  # A:4, B:2, C:2
test_cats = ['A', 'B', 'D']  # D is unseen

tr_enc, te_enc = frequency_encode(train_cats, test_cats)
print(f"Train encoded: {tr_enc}")
print(f"Test encoded:  {te_enc}")

In [ ]:
tr_enc, te_enc = frequency_encode(train_cats, test_cats)
assert tr_enc is not None, "Should return train encodings"
assert len(tr_enc) == len(train_cats), "Train encoding length mismatch"
assert abs(tr_enc[0] - 4/8) < 1e-6, "'A' frequency should be 4/8 = 0.5"
assert te_enc[-1] == 0.0, "Unseen category 'D' should get frequency 0.0"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def frequency_encode(train_categories, test_categories):
    n = len(train_categories)
    from collections import Counter
    counts = Counter(train_categories)
    freq_map = {cat: count / n for cat, count in counts.items()}
    train_enc = np.array([freq_map[c] for c in train_categories])
    test_enc = np.array([freq_map.get(c, 0.0) for c in test_categories])
    return train_enc, test_enc
```
</details>